# EEG Epilepsy – All Channel Combinations × 4 Models

Trains one of four model architectures for every possible subset of the 14 EEG channels.

**Models available** (set `MODEL_NAME` in Cell 1):
| Key | Architecture |
|---|---|
| `'CNN'` | 1-D CNN (5 Conv blocks → GAP → Dense) |
| `'ChronoNet'` | 3 inception-style Conv blocks → 4 skip-connected GRUs |
| `'CNN_LSTM'` | Deep CNN (Conv→Pool stack) → 2 LSTMs → Dense head |
| `'IC_RNN'` | Inception Conv blocks → 4 plain GRUs (no skip connections) |

**Structure**
1. Imports & Global Settings
2. Load Pre-extracted Features
3. Model Builders
4. Helper Functions
5. Main Training Loop
6. Save Metrics to Excel
7. Summary Plots


## Cell 1 – Imports & Global Settings

In [1]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import itertools
import random
import time
from itertools import combinations

# ── Numerical / data ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Plotting ────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
from matplotlib import pyplot as plt

# ── Scikit-learn ────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, cohen_kappa_score

# ── TensorFlow / Keras ──────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.layers import (
    Conv1D, BatchNormalization, LeakyReLU,
    MaxPool1D, AveragePooling1D, GlobalAveragePooling1D,
    Dense, Dropout, LSTM, GRU, concatenate, Input,
)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.backend import clear_session
from tensorflow.keras.callbacks import EarlyStopping
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  >>>  CHOOSE YOUR MODEL HERE  <<<                                       ║
# ║  Options: 'CNN'  |  'ChronoNet'  |  'CNN_LSTM'  |  'IC_RNN'            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
MODEL_NAME = 'CNN_LSTM'      # <── change this

# ── Hyper-parameters ────────────────────────────────────────────────────────
EPOCHS      = 65
BATCH_SIZE  = 128
LR          = 0.0006
TEST_SIZE   = 0.2
RANDOM_SEED = 42

# ── Channel meta-data ───────────────────────────────────────────────────────
ALL_CHANNELS  = list(range(1, 15))   # 1-based indices
CHANNEL_NAMES = [
    'AF3','AF4','F3','F4','F7','F8',
    'FC5','FC6','O1','O2','P7','P8','T7','T8'
]

# ── Output root folder ──────────────────────────────────────────────────────
OUT_ROOT = f'outputs_{MODEL_NAME}'
os.makedirs(OUT_ROOT, exist_ok=True)

assert MODEL_NAME in ('CNN', 'ChronoNet', 'CNN_LSTM', 'IC_RNN'), \
    f"Unknown MODEL_NAME '{MODEL_NAME}'. Choose from: CNN, ChronoNet, CNN_LSTM, IC_RNN"

print('Setup complete.')
print(f'TensorFlow version : {tf.__version__}')
print(f'Selected model     : {MODEL_NAME}')
print(f'Output root        : {OUT_ROOT}/')


Setup complete.
TensorFlow version : 2.21.0
Selected model     : CNN_LSTM
Output root        : outputs_CNN_LSTM/


## Cell 2 – Load Pre-extracted Features

In [2]:
FEATURE_DIR    = 'guinea-bissau_features'
FEATURE_PREFIX = 'features_guinea-bissau_channel_'
LABELS_FILE    = f'{FEATURE_DIR}/labels_groups_Guinea-Bissau.xlsx'

for its, j in enumerate(range(1, 15)):
    csv_path = f'{FEATURE_DIR}/{FEATURE_PREFIX}{j-1}.csv'
    df1 = pd.read_csv(csv_path, header=0, index_col=0).values   # (N, 640)
    if its == 0:
        df1a = df1[..., np.newaxis]
    else:
        df1a = np.concatenate((df1a, df1[..., np.newaxis]), axis=2)

print(f'Feature array shape : {df1a.shape}')   # (N, 640, 14)

df2 = pd.read_excel(LABELS_FILE, sheet_name='label_array',  header=0, index_col=0)
df3 = pd.read_excel(LABELS_FILE, sheet_name='group_array',  header=0, index_col=0)

print(f'Labels shape        : {df2.shape}')
print(f'Class balance:\n{df2.iloc[:,0].value_counts()}')


Feature array shape : (7456, 640, 14)
Labels shape        : (7456, 1)
Class balance:
0
1    3995
0    3461
Name: count, dtype: int64


## Cell 3 – Model Builders

Each function accepts `n_channels` (number of EEG channels in the current
subset) and `lr`, and returns a **compiled** Keras model.

In [3]:
# ────────────────────────────────────────────────────────────────────────────
# 1.  CNN  (original notebook architecture)
# ────────────────────────────────────────────────────────────────────────────

def build_CNN(n_channels: int, lr: float = LR) -> Sequential:
    clear_session()

    model = Sequential([
        tf.keras.Input(shape=(640, n_channels)),
        Conv1D(5, 3, strides=1, name='Conv1D_1'),
        BatchNormalization(name='BN_1'),
        LeakyReLU(name='LReLU_1'),
        MaxPool1D(2, 2, name='MaxPool_1'),

        Conv1D(5, 3, strides=1, name='Conv1D_2'),
        LeakyReLU(name='LReLU_2'),
        MaxPool1D(2, 2, name='MaxPool_2'),
        Dropout(0.2, name='Dropout_1'),

        Conv1D(5, 3, strides=1, name='Conv1D_3'),
        LeakyReLU(name='LReLU_3'),
        AveragePooling1D(2, 2, name='AvgPool_1'),
        Dropout(0.2, name='Dropout_2'),

        Conv1D(5, 3, strides=1, name='Conv1D_4'),
        LeakyReLU(name='LReLU_4'),
        AveragePooling1D(2, 2, name='AvgPool_2'),

        Conv1D(5, 3, strides=1, name='Conv1D_5'),
        LeakyReLU(name='LReLU_5'),

        GlobalAveragePooling1D(name='GAP'),
        Dense(1, activation='sigmoid', name='Output'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )

    return model

# ────────────────────────────────────────────────────────────────────────────
# 2.  ChronoNet  (inception-style Conv blocks + skip-connected GRUs)
# ────────────────────────────────────────────────────────────────────────────
def build_ChronoNet(n_channels: int, lr: float = LR) -> Model:
    clear_session()

    def _block(x, layer_no):
        c1 = Conv1D(32, 2, strides=2, activation='relu', padding='causal',
                    name=f'Conv1D_A_{layer_no}')(x)
        c2 = Conv1D(32, 4, strides=2, activation='relu', padding='causal',
                    name=f'Conv1D_B_{layer_no}')(x)
        c3 = Conv1D(32, 8, strides=2, activation='relu', padding='causal',
                    name=f'Conv1D_C_{layer_no}')(x)
        return concatenate([c1, c2, c3], axis=2, name=f'Concat_{layer_no}')

    inp    = Input(shape=(640, n_channels), name='InputLayer')
    b1     = _block(inp, 1)
    b2     = _block(b1,  2)
    b3     = _block(b2,  3)

    g1     = GRU(32, activation='tanh', return_sequences=True, name='GRU_1')(b3)
    g2     = GRU(32, activation='tanh', return_sequences=True, name='GRU_2')(g1)
    g2a    = concatenate([g1, g2], axis=2, name='Concat_4')
    g3     = GRU(32, activation='tanh', return_sequences=True, name='GRU_3')(g2a)
    g3a    = concatenate([g1, g2, g3], name='Concat_5')
    g4     = GRU(32, activation='tanh', name='GRU_4')(g3a)
    out    = Dense(1, activation='sigmoid', name='Output')(g4)

    model  = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model


# ────────────────────────────────────────────────────────────────────────────
# 3.  CNN-LSTM  (deep CNN stack + 2 LSTM layers)
# ────────────────────────────────────────────────────────────────────────────
def build_CNN_LSTM(n_channels: int, lr: float = LR) -> Model:
    clear_session()

    inp  = Input(shape=(640, n_channels), name='InputLayer')
    x    = Conv1D(64,   3, strides=1, activation='relu', padding='causal', name='Conv1D_1')(inp)
    x    = MaxPool1D(2, 2, name='MaxPool_1')(x)
    x    = Conv1D(128,  3, strides=1, activation='relu', padding='causal', name='Conv1D_2')(x)
    x    = Conv1D(512,  3, strides=1, activation='relu', padding='causal', name='Conv1D_3')(x)
    x    = Conv1D(1024, 3, strides=1, activation='relu', padding='causal', name='Conv1D_4')(x)
    x    = Dense(256, activation='relu', name='Dense_1')(x)
    x    = Dropout(0.5, name='Dropout_1')(x)
    x, *_= LSTM(64, return_sequences=True, return_state=True, name='LSTM_1')(x)
    x    = LSTM(64, name='LSTM_2')(x)
    x    = Dense(256, activation='relu', name='Dense_2')(x)
    x    = Dense(128, activation='relu', name='Dense_3')(x)
    x    = Dense(64,  activation='relu', name='Dense_4')(x)
    out  = Dense(1, activation='sigmoid', name='Output')(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model


# ────────────────────────────────────────────────────────────────────────────
# 4.  IC-RNN  (inception Conv blocks + 4 plain GRUs, no skip connections)
# ────────────────────────────────────────────────────────────────────────────
def build_IC_RNN(n_channels: int, lr: float = LR) -> Model:
    clear_session()

    def _block(x, layer_no):
        c1 = Conv1D(32, 2, strides=2, activation='relu', padding='causal',
                    name=f'Conv1D_A_{layer_no}')(x)
        c2 = Conv1D(32, 4, strides=2, activation='relu', padding='causal',
                    name=f'Conv1D_B_{layer_no}')(x)
        c3 = Conv1D(32, 8, strides=2, activation='relu', padding='causal',
                    name=f'Conv1D_C_{layer_no}')(x)
        return concatenate([c1, c2, c3], axis=2, name=f'Concat_{layer_no}')

    inp = Input(shape=(640, n_channels), name='InputLayer')
    b1  = _block(inp, 1)
    b2  = _block(b1,  2)
    b3  = _block(b2,  3)

    g1  = GRU(32, activation='tanh', return_sequences=True, name='GRU_1')(b3)
    g2  = GRU(32, activation='tanh', return_sequences=True, name='GRU_2')(g1)
    g3  = GRU(32, activation='tanh', return_sequences=True, name='GRU_3')(g2)
    g4  = GRU(32, activation='tanh', name='GRU_4')(g3)
    out = Dense(1, activation='sigmoid', name='Output')(g4)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model


# ── Dispatch table ───────────────────────────────────────────────────────────
_BUILDERS = {
    'CNN':       build_CNN,
    'ChronoNet': build_ChronoNet,
    'CNN_LSTM':  build_CNN_LSTM,
    'IC_RNN':    build_IC_RNN,
}

def build_model(n_channels: int, lr: float = LR) -> Model:
    """Build and return the model selected by MODEL_NAME."""
    return _BUILDERS[MODEL_NAME](n_channels, lr)


print('Model builders defined.')
print(f'Active builder     : build_{MODEL_NAME}()')


Model builders defined.
Active builder     : build_CNN_LSTM()


## Cell 4 – Helper Functions

In [4]:
# ────────────────────────────────────────────────────────────────────────────
# Reproducibility
# ────────────────────────────────────────────────────────────────────────────
def reset_seeds(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()


# ────────────────────────────────────────────────────────────────────────────
# Label helpers
# ────────────────────────────────────────────────────────────────────────────
def make_combo_label(kept_indices: list, n_total: int = 14) -> dict:
    kept_names    = [CHANNEL_NAMES[i-1] for i in kept_indices]
    dropped_idx   = [j for j in ALL_CHANNELS if j not in kept_indices]
    dropped_names = [CHANNEL_NAMES[j-1] for j in dropped_idx]
    n_keep = len(kept_indices)
    n_drop = len(dropped_idx)

    if n_drop == 0:
        title = 'All 14 channels'
    elif n_keep >= 8:
        title = f'Dropped: {", ".join(dropped_names)}'
    else:
        title = f'Kept: {", ".join(kept_names)}'

    drop_str = '_'.join(dropped_names) if dropped_names else 'none'
    fname    = f'keep{n_keep:02d}_drop_{drop_str}'

    return {
        'title': title, 'fname': fname,
        'n_keep': n_keep, 'n_drop': n_drop,
        'kept_names': kept_names, 'dropped_names': dropped_names,
    }


# ────────────────────────────────────────────────────────────────────────────
# Per-combination figure saver
# ────────────────────────────────────────────────────────────────────────────
def save_figures_for_combo(history, y_test, y_pred,
                            label_info: dict, out_dir: str) -> dict:
    os.makedirs(out_dir, exist_ok=True)
    title  = label_info['title']
    fname  = label_info['fname']
    fw, fs = 'bold', 12

    y_pred_bin  = np.round(y_pred).astype(int).squeeze()
    cm          = confusion_matrix(y_test, y_pred_bin)
    cm_norm     = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    auc         = roc_auc_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    cks         = cohen_kappa_score(y_test, y_pred_bin)

    tn, fp, fn, tp = cm.ravel()
    precision = 100 * tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = 100 * tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) \
                if (precision + recall) > 0 else 0.0
    
    # getting the best val_acc due to callback
    best_epoch = np.argmax(history.history['val_accuracy'])

    val_acc  = history.history['val_accuracy'][best_epoch] * 100
    val_loss = history.history['val_loss'][best_epoch]


    # 1. Accuracy curve
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(history.history['accuracy'],     color='#00ABCD', label='Train')
    ax.plot(history.history['val_accuracy'], color='#121212', label='Test')
    ax.set_title(f'Accuracy – {title}', fontweight=fw, fontsize=fs)
    ax.set_xlabel('Epoch', fontweight=fw, fontsize=fs)
    ax.set_ylabel('Accuracy', fontweight=fw, fontsize=fs)
    ax.set_xlim(0, EPOCHS); ax.set_ylim(0, 1)
    ax.set_xticks(np.arange(0, EPOCHS + 1, 20))
    ax.legend(prop={'weight': fw}); ax.tick_params(labelsize=fs)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, f'{fname}_accuracy_curve.png'), dpi=150)
    plt.close(fig)

    # 2. Loss curve
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(history.history['loss'],     color='#00ABCD', label='Train')
    ax.plot(history.history['val_loss'], color='#121212', label='Test')
    ax.set_title(f'Loss – {title}', fontweight=fw, fontsize=fs)
    ax.set_xlabel('Epoch', fontweight=fw, fontsize=fs)
    ax.set_ylabel('Binary Cross-entropy', fontweight=fw, fontsize=fs)
    ax.set_xlim(0, EPOCHS); ax.set_ylim(0, 1)
    ax.set_xticks(np.arange(0, EPOCHS + 1, 20))
    ax.legend(prop={'weight': fw}); ax.tick_params(labelsize=fs)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, f'{fname}_loss_curve.png'), dpi=150)
    plt.close(fig)

    # 3. Confusion Matrix
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.matshow(cm, cmap=plt.cm.binary)
    threshold = (cm.max() + cm.min()) / 2.0
    for i_cm, j_cm in itertools.product(range(2), range(2)):
        ax.text(j_cm, i_cm, f'{cm_norm[i_cm, j_cm]*100:.1f}%',
                ha='center', va='center',
                color='white' if cm[i_cm, j_cm] > threshold else 'black',
                fontweight=fw, fontsize=fs)
    ax.set_title(f'Confusion Matrix\n{title}', fontweight=fw, fontsize=fs)
    ax.set_xlabel('Predicted Label', fontweight=fw, fontsize=fs)
    ax.set_ylabel('True Label', fontweight=fw, fontsize=fs)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.xaxis.set_label_position('bottom'); ax.xaxis.tick_bottom()
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, f'{fname}_confusion_matrix.png'), dpi=150)
    plt.close(fig)

    # 4. ROC curve
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.plot(fpr, tpr, color='#000000', label=f'AUC = {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    ax.set_title(f'ROC – {title}', fontweight=fw, fontsize=fs)
    ax.set_xlabel('False Positive Rate', fontweight=fw, fontsize=fs)
    ax.set_ylabel('True Positive Rate', fontweight=fw, fontsize=fs)
    ax.set_xticks(np.linspace(0, 1, 3)); ax.set_yticks(np.linspace(0, 1, 3))
    ax.legend(prop={'weight': fw}); ax.tick_params(labelsize=fs)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, f'{fname}_roc_curve.png'), dpi=150)
    plt.close(fig)

    return {
        'model':     MODEL_NAME,
        'title':     title,
        'fname':     fname,
        'n_keep':    label_info['n_keep'],
        'kept':      ', '.join(label_info['kept_names']),
        'dropped':   ', '.join(label_info['dropped_names']) or 'none',
        'val_acc':   val_acc,
        'val_loss':  val_loss,
        'auc':       auc,
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
        'cohen_kappa': cks,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
    }


print('Helper functions defined.')


Helper functions defined.


## Cell 5 – Main Training Loop

Iterates over every non-empty subset of the 14 channels (2^14 − 1 = 16 383).

> Set `MAX_COMBOS = None` to run all combinations.

In [5]:
MAX_COMBOS = None     # set to None for a full run

all_results   = []
combo_counter = 0

for n_exclude in range(2,3):                  #(0,14)
    n_keep = 14 - n_exclude

    for excl in combinations(ALL_CHANNELS, n_exclude):

        if MAX_COMBOS is not None and combo_counter >= MAX_COMBOS:
            print(f'Reached MAX_COMBOS = {MAX_COMBOS}. Stopping.')
            break

        my_list    = [j for j in ALL_CHANNELS if j not in excl]
        label_info = make_combo_label(my_list)
        combo_counter += 1

        print(f'[{combo_counter:5d}] model={MODEL_NAME} | keep={n_keep:2d} | {label_info["title"]}',
              flush=True)

        combo_dir = os.path.join(OUT_ROOT, f'keep{n_keep:02d}', label_info['fname'])
        os.makedirs(combo_dir, exist_ok=True)

        # ── Prepare data ────────────────────────────────────────────────────
        data_array  = df1a[..., np.array(my_list) - 1]   # (N, 640, n_keep)
        label_array = df2.iloc[:, 0].values

        X_train, X_test, y_train, y_test = train_test_split(
            data_array, label_array,
            test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=label_array,
        )

        sc      = StandardScaler()
        X_train = sc.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
        X_test  = sc.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

        # ── Build & train ────────────────────────────────────────────────────
        reset_seeds(RANDOM_SEED)
        model = build_model(n_channels=n_keep, lr=LR)

        t0 = time.time()

        # moving it near fit(), the goal was to ensure a fresh callback is created for every iteration:
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=8,
            restore_best_weights=True,
            verbose=1,
            mode='max'
        )

        history = model.fit(
            tf.constant(X_train), tf.constant(y_train),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(tf.constant(X_test), y_test),
            callbacks=[early_stop],
            verbose=1
        )

        train_time = time.time() - t0

        # ── Evaluate ─────────────────────────────────────────────────────────
        y_pred = model.predict(X_test, verbose=0)

        # ── Save figures & collect metrics ────────────────────────────────────
        metrics = save_figures_for_combo(
            history=history,
            y_test=y_test,
            y_pred=y_pred,
            label_info=label_info,
            out_dir=combo_dir,
        )
        metrics['train_time_s'] = round(train_time, 1)

        # ── Save training history CSV ─────────────────────────────────────────
        pd.DataFrame(history.history).to_csv(
            os.path.join(combo_dir, f'{label_info["fname"]}_history.csv'),
            index_label='epoch',
        )

        all_results.append(metrics)
        print(f'        val_acc={metrics["val_acc"]:.1f}%  '
              f'AUC={metrics["auc"]:.3f}  '
              f'F1={metrics["f1"]:.1f}%  '
              f'κ={metrics["cohen_kappa"]:.3f}  '
              f't={train_time:.0f}s',
              flush=True)

    else:
        continue
    break

print(f'\nDone. Combinations run: {combo_counter}')


[    1] model=CNN_LSTM | keep=12 | Dropped: AF3, AF4

Epoch 1/65
47/47 ━━━━━━━━━━━━━━━━━━━━ 41s 826ms/step - accuracy: 0.5741 - loss: 0.6701 - val_accuracy: 0.6957 - val_loss: 0.6117
Epoch 2/65
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 807ms/step - accuracy: 0.6873 - loss: 0.6016 - val_accuracy: 0.7306 - val_loss: 0.5797
Epoch 3/65
24/47 ━━━━━━━━━━━━━━━━━━━━ 16s 714ms/step - accuracy: 0.7401 - loss: 0.5585

KeyboardInterrupt: 

## Cell 6 – Save All Metrics to Excel

In [ ]:
results_df   = pd.DataFrame(all_results)
summary_path = os.path.join(OUT_ROOT, f'{MODEL_NAME}_AllCombos_Summary.xlsx')

with pd.ExcelWriter(summary_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='All Results', index=False)
    results_df.sort_values('val_acc', ascending=False).to_excel(
        writer, sheet_name='Sorted by Accuracy', index=False)
    results_df.sort_values('auc', ascending=False).to_excel(
        writer, sheet_name='Sorted by AUC', index=False)
    results_df.sort_values('f1', ascending=False).to_excel(
        writer, sheet_name='Sorted by F1', index=False)
    results_df.sort_values('cohen_kappa', ascending=False).to_excel(
        writer, sheet_name='Sorted by Kappa', index=False)

print(f'Summary saved → {summary_path}')
results_df.head()


Summary saved → outputs_IC_RNN\IC_RNN_AllCombos_Summary.xlsx


,model,title,fname,n_keep,kept,dropped,val_acc,val_loss,auc,precision,recall,f1,cohen_kappa,tn,fp,fn,tp,train_time_s
0,IC_RNN,"Dropped: AF3, AF4",keep12_drop_AF3_AF4,12,"F3, F4, F7, F8, FC5, FC6, O1, O2, P7, P8, T7, T8","AF3, AF4",93.230563,0.201065,0.982594,95.207254,91.989987,93.570974,0.864275,656,37,64,735,184.2
1,IC_RNN,"Dropped: AF3, F3",keep12_drop_AF3_F3,12,"AF4, F4, F7, F8, FC5, FC6, O1, O2, P7, P8, T7, T8","AF3, F3",93.096513,0.194217,0.978038,93.718593,93.366708,93.542320,0.861270,643,50,53,746,168.4
2,IC_RNN,"Dropped: AF3, F4",keep12_drop_AF3_F4,12,"AF4, F3, F7, F8, FC5, FC6, O1, O2, P7, P8, T7, T8","AF3, F4",92.895442,0.196391,0.978115,94.709677,91.864831,93.265565,0.857515,652,41,65,734,130.4
3,IC_RNN,"Dropped: AF3, F7",keep12_drop_AF3_F7,12,"AF4, F3, F4, F8, FC5, FC6, O1, O2, P7, P8, T7, T8","AF3, F7",94.101876,0.169723,0.985426,93.195626,95.994994,94.574599,0.881166,637,56,32,767,163.1
4,IC_RNN,"Dropped: AF3, F8",keep12_drop_AF3_F8,12,"AF4, F3, F4, F7, FC5, FC6, O1, O2, P7, P8, T7, T8","AF3, F8",94.705093,0.179727,0.985913,95.569620,94.493116,95.028320,0.893656,658,35,44,755,204.2


## Cell 7 – Summary Plots

In [ ]:
fw, fs = 'bold', 12

# ── 7a. Box-plots: metric distribution per subset size ───────────────────────
for metric, ylabel, plot_title in [
    ('val_acc', 'Validation Accuracy (%)', 'Accuracy by Subset Size'),
    ('auc',     'AUC',                     'AUC by Subset Size'),
    ('f1',      'F1 Score (%)',            'F1 Score by Subset Size'),
]:
    grouped = [results_df.loc[results_df['n_keep'] == k, metric].values
               for k in range(1, 15)]
    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(grouped, patch_artist=True,
                    boxprops=dict(facecolor='#00ABCD', color='#121212'),
                    medianprops=dict(color='#FF4500', linewidth=2))
    ax.set_title(f'{MODEL_NAME} – {plot_title}', fontweight=fw, fontsize=fs)
    ax.set_xlabel('Number of Channels Kept', fontweight=fw, fontsize=fs)
    ax.set_ylabel(ylabel, fontweight=fw, fontsize=fs)
    ax.set_xticks(range(1, 15)); ax.set_xticklabels(range(1, 15), fontweight=fw)
    ax.tick_params(labelsize=fs)
    plt.tight_layout()
    out_path = os.path.join(OUT_ROOT, f'summary_boxplot_{metric}.png')
    fig.savefig(out_path, dpi=150); plt.close(fig)
    print(f'Saved: {out_path}')


# ── 7b. Top-10 by accuracy ───────────────────────────────────────────────────
top10 = results_df.sort_values('val_acc', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_title(f'{MODEL_NAME} – Top-10 Combinations by Validation Accuracy',
             fontweight=fw, fontsize=fs)
ax.set_xlabel('Validation Accuracy (%)', fontweight=fw, fontsize=fs)
hbars = ax.barh(range(len(top10)), top10['val_acc'].values, color='#121212')
ax.bar_label(hbars, fmt=lambda x: f'{x:.1f}%', fontsize=fs, fontweight=fw)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10['title'].values, fontweight=fw, fontsize=10)
ax.set_xlim(0, 110)
plt.tight_layout()
out_path = os.path.join(OUT_ROOT, 'summary_top10_accuracy.png')
fig.savefig(out_path, dpi=150); plt.close(fig)
print(f'Saved: {out_path}')


# ── 7c. Channel importance (Δ accuracy) ─────────────────────────────────────
mean_acc_with    = {}
mean_acc_without = {}
for ch in CHANNEL_NAMES:
    mean_acc_with[ch]    = results_df[results_df['kept'].str.contains(ch)]['val_acc'].mean()
    mean_acc_without[ch] = results_df[~results_df['kept'].str.contains(ch)]['val_acc'].mean()

importance_df = pd.DataFrame({
    'Channel':          CHANNEL_NAMES,
    'Mean Acc With':    [mean_acc_with[c]    for c in CHANNEL_NAMES],
    'Mean Acc Without': [mean_acc_without[c] for c in CHANNEL_NAMES],
})
importance_df['Delta'] = importance_df['Mean Acc With'] - importance_df['Mean Acc Without']
importance_df = importance_df.sort_values('Delta', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#121212' if d >= 0 else '#CC0000' for d in importance_df['Delta']]
hbars = ax.barh(range(len(importance_df)), importance_df['Delta'].values, color=colors)
ax.bar_label(hbars, fmt=lambda x: f'{x:+.2f}%', fontsize=fs, fontweight=fw)
ax.set_title(f'{MODEL_NAME} – Channel Importance\n(Mean Acc with − without)',
             fontweight=fw, fontsize=fs)
ax.set_xlabel('Δ Accuracy (%)', fontweight=fw, fontsize=fs)
ax.set_yticks(range(len(importance_df)))
ax.set_yticklabels(importance_df['Channel'].values, fontweight=fw, fontsize=fs)
ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
plt.tight_layout()
out_path = os.path.join(OUT_ROOT, 'summary_channel_importance.png')
fig.savefig(out_path, dpi=150); plt.close(fig)
print(f'Saved: {out_path}')

importance_df.to_csv(os.path.join(OUT_ROOT, 'channel_importance.csv'), index=False)
print('All summary plots saved.')
importance_df.sort_values('Delta', ascending=False)


Saved: outputs_IC_RNN\summary_boxplot_val_acc.png
Saved: outputs_IC_RNN\summary_boxplot_auc.png
Saved: outputs_IC_RNN\summary_boxplot_f1.png
Saved: outputs_IC_RNN\summary_top10_accuracy.png
Saved: outputs_IC_RNN\summary_channel_importance.png
All summary plots saved.


,Channel,Mean Acc With,Mean Acc Without,Delta
3,F4,93.060024,92.627347,0.432677
9,O2,93.107685,92.740772,0.366913
13,T8,93.091359,92.838730,0.252629
1,AF4,93.085344,92.874819,0.210525
11,P8,93.081048,92.900598,0.180450
10,P7,93.073314,92.946999,0.126315
12,T7,93.056128,93.050114,0.006015
7,FC6,93.054410,93.060425,-0.006015
2,F3,93.054811,93.096513,-0.041702
8,O1,93.035506,93.173850,-0.138344
